# GAN Example 3 — A 2D Toy GAN (watch a GAN learn a distribution)

Images make it hard to *see* what a GAN is really doing. This example strips the problem
down to **2D points** so you can plot the real and fake distributions side by side and
literally watch the generator's cloud crawl onto the target.

## The target

Our 'real data' is a ring of **8 Gaussian blobs** (a classic GAN benchmark). The
generator must learn to cover *all eight* — if it only learns a few, you are seeing
**mode collapse**, the most famous GAN failure mode, in plain sight.

Both networks are tiny MLPs, so this trains in seconds on CPU.

> Tested with `tensorflow==2.1`.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
import matplotlib.pyplot as plt

np.random.seed(0)
tf.random.set_seed(0)
print('TensorFlow', tf.__version__)

## 1. The real data — 8 Gaussians in a ring

In [ ]:
NOISE_DIM = 2
centers = np.array([[np.cos(t), np.sin(t)] for t in np.linspace(0, 2*np.pi, 8, endpoint=False)],
                   dtype='float32') * 2.0

def sample_real(n):
    idx = np.random.randint(0, 8, size=n)
    return (centers[idx] + 0.1 * np.random.randn(n, 2)).astype('float32')

real = sample_real(2000)
plt.figure(figsize=(4, 4))
plt.scatter(real[:, 0], real[:, 1], s=4)
plt.title('Target distribution (8 Gaussians)'); plt.axis('equal'); plt.show()

## 2. Tiny generator & discriminator

Both are 2-hidden-layer MLPs. The generator maps a 2D noise vector to a 2D point; the
discriminator maps a 2D point to a realness logit.

In [ ]:
def mlp(out, last=None):
    return tf.keras.Sequential([
        layers.Dense(128, activation='relu'),
        layers.Dense(128, activation='relu'),
        layers.Dense(out, activation=last),
    ])

generator = mlp(2)            # noise -> point
discriminator = mlp(1)        # point -> logit
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)
g_opt = tf.keras.optimizers.Adam(1e-3, beta_1=0.5)
d_opt = tf.keras.optimizers.Adam(1e-3, beta_1=0.5)

In [ ]:
@tf.function
def train_step(real_batch):
    bsz = tf.shape(real_batch)[0]
    noise = tf.random.normal([bsz, NOISE_DIM])
    with tf.GradientTape() as gt, tf.GradientTape() as dt:
        fake = generator(noise, training=True)
        real_out = discriminator(real_batch, training=True)
        fake_out = discriminator(fake, training=True)
        d_loss = (cross_entropy(tf.ones_like(real_out), real_out) +
                  cross_entropy(tf.zeros_like(fake_out), fake_out))
        g_loss = cross_entropy(tf.ones_like(fake_out), fake_out)
    d_opt.apply_gradients(zip(dt.gradient(d_loss, discriminator.trainable_variables),
                              discriminator.trainable_variables))
    g_opt.apply_gradients(zip(gt.gradient(g_loss, generator.trainable_variables),
                              generator.trainable_variables))
    return g_loss, d_loss

In [ ]:
def plot_compare(epoch):
    fake = generator(tf.random.normal([2000, NOISE_DIM]), training=False).numpy()
    plt.figure(figsize=(4, 4))
    plt.scatter(real[:, 0], real[:, 1], s=4, alpha=0.3, label='real')
    plt.scatter(fake[:, 0], fake[:, 1], s=4, alpha=0.3, label='fake')
    plt.legend(); plt.axis('equal'); plt.title(f'Epoch {epoch}'); plt.show()

## 3. Train

We draw fresh real samples each step (the target is a generator, not a fixed set).
Watch the orange (fake) cloud spread to cover all eight blobs.

In [ ]:
STEPS = 5000
for step in range(1, STEPS + 1):
    g_loss, d_loss = train_step(tf.constant(sample_real(256)))
    if step % 1000 == 0 or step == 1:
        print(f'Step {step:5d}/{STEPS}  gen_loss={float(g_loss):.3f}  disc_loss={float(d_loss):.3f}')
        plot_compare(step)

### Things to try

- Shrink the generator (e.g. 16 units) to *induce* mode collapse and watch it cover only
  a few blobs.
- Change the target to a spiral or two moons.
- Train the discriminator multiple steps per generator step and see the effect.
- This 2D setup is the ideal sandbox to test stabilisers like **WGAN** or gradient
  penalty — easy to see if they cure collapse.